# CS0502 · ModelScope PAI-DSW 快速开始

本 Notebook 只负责运行课程 example、Lab 和交互可视化。知识学习、预测、提示和复盘仍在你自己电脑上的 OpenCode 中完成。

建议流程：**先在 OpenCode 预测 → 再运行下面的 helper → 把 `[CS0502_RESULT]` 摘要复制回 OpenCode**。云端不需要也不应保存 Qwen API Key。

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(f"课程环境要求 Python 3.11+，当前为 {platform.python_version()}；请重新选择 Python 3.11 或更高版本的 PAI-DSW 镜像。")

WORKSPACE = Path("/mnt/workspace")
if not WORKSPACE.is_dir():
    raise RuntimeError("未找到 /mnt/workspace；请使用支持持久化存储的 PAI-DSW Notebook，而不是 EAIS。")
if shutil.which("git") is None:
    raise RuntimeError("当前镜像缺少 git，请联系课程组更换教学镜像。")

COURSE_ROOT = WORKSPACE / "CS0502"
REPO_URL = "https://github.com/jinyh/cs-intro-knowledge-expansion.git"
print(f"平台: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"课程目录: {COURSE_ROOT}")

In [ ]:
def checked(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if (COURSE_ROOT / ".git").is_dir():
    checked(["git", "pull", "--ff-only"], cwd=COURSE_ROOT)
elif COURSE_ROOT.exists() and any(COURSE_ROOT.iterdir()):
    raise RuntimeError(f"{COURSE_ROOT} 已存在且不是课程 Git 仓库；请先在文件浏览器中改名后重试。")
else:
    try:
        checked(["git", "clone", "--depth", "1", REPO_URL, str(COURSE_ROOT)])
    except subprocess.CalledProcessError as error:
        raise RuntimeError("无法从 GitHub 获取课程仓库。请下载课程组发布的 ZIP，上传并解压为 /mnt/workspace/CS0502。") from error

print("课程仓库已就绪。student-work/ 被 Git 忽略，更新仓库不会覆盖学生副本。")

In [ ]:
VENV = COURSE_ROOT / ".venv"
if not (VENV / "bin" / "python").is_file():
    checked([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)])

COURSE_PYTHON = VENV / "bin" / "python"
checked([str(COURSE_PYTHON), "-m", "pip", "install", "-r", str(COURSE_ROOT / "code" / "requirements.txt"), "pytest>=8.3"])

COURSE_ENV = os.environ.copy()
COURSE_ENV["CS0502_RUNTIME"] = "modelscope"
COURSE_ENV["MPLBACKEND"] = "Agg"
print(f"课程 Python: {COURSE_PYTHON}")

In [ ]:
from datetime import datetime, timezone
import html
import re
import zipfile
from IPython.display import HTML, SVG, display

RUNNER = COURSE_ROOT / "code" / "runner.py"

def _safe_name(value, suffix=None):
    name = Path(value).name
    if name != value or (suffix and not name.endswith(suffix)):
        raise ValueError(f"无效资源名: {value}")
    return name

def _first_evidence(result):
    lines = [line.strip() for line in (result.stdout + "\n" + result.stderr).splitlines() if line.strip()]
    preferred = next((line for line in lines if re.search(r"FAILED|ERROR|AssertionError|被拒绝|终止", line, re.I)), None)
    return (preferred or (lines[-1] if lines else "无控制台输出"))[:300].replace("=", "：")

def _run_runner(kind, resource_id, arguments):
    result = subprocess.run(
        [str(COURSE_PYTHON), str(RUNNER), *arguments],
        cwd=COURSE_ROOT, env=COURSE_ENV, text=True, capture_output=True, check=False,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)
    status = "passed" if result.returncode == 0 else "needs-work"
    print("\n[CS0502_RESULT]")
    print(f"kind={kind}")
    print(f"id={resource_id}")
    print(f"status={status}")
    print(f"returncode={result.returncode}")
    print(f"first_evidence={_first_evidence(result)}")
    print("[/CS0502_RESULT]")
    return result

def run_example(filename):
    filename = _safe_name(filename, ".py")
    path = COURSE_ROOT / "code" / "examples" / filename
    if not path.is_file():
        raise FileNotFoundError(f"不存在的 example: {filename}")
    return _run_runner("example", filename, ["run", str(path.relative_to(COURSE_ROOT))])

def init_lab(lab_id):
    lab_id = _safe_name(lab_id)
    return _run_runner("lab-init", lab_id, ["lab", "init", lab_id])

def test_lab(lab_id):
    lab_id = _safe_name(lab_id)
    return _run_runner("lab-test", lab_id, ["lab", "test", lab_id])

def show_visualization(filename, height=680):
    filename = _safe_name(filename, ".html")
    path = COURSE_ROOT / "code" / "visualizations" / filename
    if not path.is_file():
        raise FileNotFoundError(f"不存在的可视化: {filename}")
    source = html.escape(path.read_text(encoding="utf-8"), quote=True)
    display(HTML(f'<iframe title="{html.escape(filename)}" srcdoc="{source}" width="100%" height="{int(height)}" style="border:1px solid #ddd"></iframe>'))

def show_svg(relative_path="student-work/15-visualization.svg"):
    path = (COURSE_ROOT / relative_path).resolve()
    if COURSE_ROOT not in path.parents or not path.is_file() or path.suffix.lower() != ".svg":
        raise ValueError(f"无效或不存在的 SVG: {relative_path}")
    display(SVG(filename=str(path)))

def run_smoke_tests():
    return subprocess.run([str(COURSE_PYTHON), "-m", "pytest", "-q"], cwd=COURSE_ROOT, env=COURSE_ENV, check=False)

def export_student_work():
    source = COURSE_ROOT / "student-work"
    source.mkdir(exist_ok=True)
    backup_root = WORKSPACE / "CS0502-backups"
    backup_root.mkdir(exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%SZ")
    archive = backup_root / f"student-work-{timestamp}.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(source.rglob("*")):
            if path.is_file() and not path.is_symlink():
                bundle.write(path, path.relative_to(COURSE_ROOT))
    print(f"备份已创建: {archive}")
    return archive

print("Helper 已加载：run_example / init_lab / test_lab / show_visualization / show_svg / run_smoke_tests / export_student_work")

## 首次环境验证

下面先运行一个纯标准库 example 和一个 NumPy example，再执行仓库测试。首次安装后建议完整运行一次。

In [ ]:
run_example("06_graph_bfs_dfs.py")
run_example("14_clustering.py")

In [ ]:
run_smoke_tests()

## 日常使用

不要直接运行陌生代码。先在本机 OpenCode 输入 `/demo L05 图遍历` 或 `/lab lab-02-graph`，完成预测后再执行它给出的 helper。

Lab 示例：

```python
init_lab("lab-02-graph")
# 在 Web IDE 中编辑 /mnt/workspace/CS0502/student-work/labs/lab-02-graph/solution.py
test_lab("lab-02-graph")
```

可视化示例：

```python
show_visualization("binary_heap.html")
run_example("15_visualization.py")
show_svg()
```

每周或重要实验后运行 `export_student_work()`，再通过左侧文件浏览器下载 ZIP。